In [56]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import re
import time
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv

In [57]:
load_dotenv()
end_date = datetime.today().strftime("%Y-%m-%d")
scroll_pause = 2  # seconden tussen scrolls


In [58]:
my_twitter_email = os.getenv("my_twitter_email")
my_twitter_username = os.getenv("my_twitter_username")
my_twitter_password = os.getenv("my_twitter_password")

In [59]:
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-notifications")

In [60]:
driver = uc.Chrome(version_main=143, options=options)
time.sleep(5)

In [61]:
# Ga naar loginpagina
driver.get("https://twitter.com/login")
time.sleep(5)  # wacht tot de pagina volledig geladen is

In [62]:
username_input = driver.find_element(By.NAME, "text")
username_input.send_keys(my_twitter_username)
time.sleep(1)
username_input.send_keys(Keys.ENTER)
time.sleep(3)

In [63]:
try:
    password_input = driver.find_element(By.NAME, "password")
    password_input.send_keys(my_twitter_password)
    time.sleep(5)
    password_input.send_keys(Keys.ENTER)
    time.sleep(5)
except:
    print("Geen wachtwoord invoer vereist of al ingelogd.")

In [64]:
usernames_we_wanna_scrape = ["elonmusk", "realDonaldTrump"]

In [65]:
def twitter_handle(username : str):
    driver.get(f"https://twitter.com/{username}")
    time.sleep(7)  # wachten tot pagina laadt
    # zorg dat cookies worden ge accepteert indien nodig:
    try:
        cookie_button = driver.find_element(By.XPATH, '//button[contains(., "Accept all cookies")]')
        cookie_button.click()
        print("Cookies geaccepteerd.")
        time.sleep(2)
    except:
        print("Geen cookie-wall gevonden of al geaccepteerd.")

In [66]:
tweets_data = []

In [67]:
def human_scroll(driver, total_scroll=3000, step=300, pause=0.3):
    scrolled = 0
    while scrolled < total_scroll:
        driver.execute_script(f"window.scrollBy(0, {step});")
        scrolled += step
        time.sleep(pause)

In [68]:
def save_to_csv():
    df = pd.DataFrame(tweets_data, columns=["Date", "Username", "Content", "Replies", "Reposts", "Likes", "Bookmarks", "Views"])
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d %H:%M')
    df['Time'] = pd.to_datetime(df['Date']).dt.time
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    print(f"{len(df)} tweets opgeslagen!")
    return df

In [ ]:
# Instellingen
target_date = end_date  # Stop met scrapen bij tweets ouder dan 1 jan 2026
tweets_ids = set() 

for user in usernames_we_wanna_scrape:
    twitter_handle(username=user)
    stop_scraping = False
    print(f"Start met scrapen tot aan {target_date} van {user}...")
    while stop_scraping == False:
        if stop_scraping:
            break
            
        # Zoek alle zichtbare tweets
        articles = driver.find_elements(By.XPATH, '//article[@role="article" and @data-testid="tweet"]')
        
        for article in articles:
            try:
                # 1. Datum ophalen
                time_elem = article.find_element(By.XPATH, './/time')
                date_str = time_elem.get_attribute("datetime")
                tweet_date = datetime.fromisoformat(date_str.replace("Z", "+00:00")).replace(tzinfo=None)

                # 2. Unieke ID check
                tweet_id = article.find_element(By.XPATH, './/time/..').get_attribute('href')
                tweet_username = tweet_id.split("/")[3]
                
                print(f"date: {tweet_date}\ntweet_usernam: {tweet_username}")
                
                if tweet_id in tweets_ids:
                    continue

                if tweet_username != user:
                    print(f"username: {tweet_username} is not {user}")
                    continue

                # 3. Check pinned
                try:
                    social_context = article.find_element(By.CSS_SELECTOR, "div[data-testid='socialContext']")
                    is_pinned = "Pinned" in social_context.text
                except:
                    is_pinned = False
                    
                print(f"pinned: {is_pinned}")
                    

                # 4. Beslissing opslaan
                # Opslaan alleen als pinned of tweet van vandaag
                if is_pinned and tweet_date.strftime("%Y-%m-%d") < target_date:
                    print(f"tweet: {tweet_date} is pinned and has older date")
                    continue
                elif tweet_date.strftime("%Y-%m-%d") < target_date:
                    stop_scraping = True
                    break
                
                
                

                # 5. Tekst en statistieken ophalen
                text = article.find_element(By.XPATH, './/div[@data-testid="tweetText"]').text
                stats_group = article.find_element(By.XPATH, './/div[@role="group"]')
                label = stats_group.get_attribute("aria-label")

                def parse_stat(pattern, text):
                    match = re.search(pattern, text.lower())
                    if match:
                        return int(re.sub(r'[^\d]', '', match.group(1)))
                    return 0

                replies = parse_stat(r'(\d[\d\.,]*)\s+replies', label)
                reposts = parse_stat(r'(\d[\d\.,]*)\s+reposts', label)
                likes = parse_stat(r'(\d[\d\.,]*)\s+likes', label)
                bookmarks = parse_stat(r'(\d[\d\.,]*)\s+bookmarks', label)
                views = parse_stat(r'(\d[\d\.,]*)\s+views', label)

                # Opslaan
                tweets_data.append([tweet_date, tweet_username, text, replies, reposts, likes, bookmarks, views])
                tweets_ids.add(tweet_id)
                print(f"Opgeslagen: {tweet_date.strftime('%Y-%m-%d %H:%M')} | @{tweet_username} | Likes: {likes}")

            except Exception as e:
                # Optioneel: print(e) voor debug
                continue

        
        # Scroll naar beneden om nieuwe tweets te laden
        human_scroll(driver, total_scroll=2500, step=250, pause=0.4)
        time.sleep(scroll_pause)

print(f"Klaar! {len(tweets_data)} tweets verzameld.")

Cookies geaccepteerd.
Start met scrapen tot aan 2026-01-31 van elonmusk...
date: 2026-01-31 21:10:13
tweet_usernam: elonmusk
pinned: False
Opgeslagen: 2026-01-31 21:10 | @elonmusk | Likes: 2121
date: 2026-01-31 20:44:04
tweet_usernam: elonmusk
pinned: False
Opgeslagen: 2026-01-31 20:44 | @elonmusk | Likes: 3933
date: 2026-01-31 19:43:07
tweet_usernam: Starlink
username: Starlink is not elonmusk
date: 2026-01-31 18:16:17
tweet_usernam: AdamLowisz
username: AdamLowisz is not elonmusk
date: 2026-01-31 18:29:40
tweet_usernam: elonmusk
pinned: False
Opgeslagen: 2026-01-31 18:29 | @elonmusk | Likes: 18178
date: 2026-01-31 21:10:13
tweet_usernam: elonmusk
date: 2026-01-31 20:44:04
tweet_usernam: elonmusk
date: 2026-01-31 19:43:07
tweet_usernam: Starlink
username: Starlink is not elonmusk
date: 2026-01-31 18:16:17
tweet_usernam: AdamLowisz
username: AdamLowisz is not elonmusk
date: 2026-01-31 18:29:40
tweet_usernam: elonmusk
date: 2026-01-31 18:06:11
tweet_usernam: AdamLowisz
username: AdamLow

,Date,Username,Content,Replies,Reposts,Likes,Bookmarks,Views,Time
0,2026-01-31,elonmusk,Just the very early stages of the singularity....,679,271,2121,120,343425,21:10:00
1,2026-01-31,elonmusk,Definitely,932,569,3933,183,911588,20:44:00
2,2026-01-31,elonmusk,This is why,1855,4789,18178,814,1806902,18:29:00
3,2026-01-31,elonmusk,It has indeed somehow become even worse,1841,4155,26853,1234,3071813,18:14:00
4,2026-01-31,elonmusk,Delaware Supreme is saving the state,989,1916,13985,255,2059963,18:08:00
5,2026-01-31,elonmusk,,1663,1916,18301,464,2845506,18:06:00
6,2026-01-31,elonmusk,China electricity generation is still growing ...,3225,3866,23186,1776,5389597,18:03:00
7,2026-01-31,elonmusk,Space is huge,1908,908,8031,235,6655448,17:57:00
8,2026-01-31,elonmusk,Still remains true,2642,7091,51208,996,3984037,17:56:00
9,2026-01-31,elonmusk,Yes,2303,3749,21462,391,2877435,17:54:00


In [74]:
big_file = pd.concat([save_to_csv(), pd.read_csv("../../raw/elonmusk_tweets.csv")],ignore_index=True)

26 tweets opgeslagen!


In [75]:
big_file.set_index('Date', inplace=True)

In [76]:
big_file.head()

,Username,Content,Replies,Reposts,Likes,Bookmarks,Views,Time
Date,,,,,,,,
2026-01-31,elonmusk,Just the very early stages of the singularity....,679,271,2121,120,343425,21:10:00
2026-01-31,elonmusk,Definitely,932,569,3933,183,911588,20:44:00
2026-01-31,elonmusk,This is why,1855,4789,18178,814,1806902,18:29:00
2026-01-31,elonmusk,It has indeed somehow become even worse,1841,4155,26853,1234,3071813,18:14:00
2026-01-31,elonmusk,Delaware Supreme is saving the state,989,1916,13985,255,2059963,18:08:00


In [77]:
big_file.to_csv("../../raw/elonmusk_tweets.csv", index=True)